<a href="https://colab.research.google.com/github/Akhilrajeevp/ACL_GenderInclusiveworkshop2026/blob/main/CF_Generation_steering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForImageTextToText
from tqdm import tqdm
from sklearn.decomposition import PCA

# Config
MODEL_ID = "google/gemma-3-4b-it"
CSV_FILE = "Counterfactual Sentence Pairs_training.csv"
LAYER_ID = 16 # Target layer for steering

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)
print("Model loaded.")

Loading google/gemma-3-4b-it...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Model loaded.


In [ ]:
# 1. Load Data
df = pd.read_csv(CSV_FILE)
pairs = [(r['Biased Sentence'], r['Counterfactual Sentence'])
         for _, r in df.iterrows()
         if isinstance(r['Biased Sentence'], str)]

# 2. Helper to find the correct text layer in Gemma 3
def get_layer(model, idx):
    # Helper to search within a specific root module
    def find_layers_recursive(module):
        # Check common names for layer containers
        for name in ['layers', 'h', 'blocks', 'decoder']:
            if hasattr(module, name):
                attr = getattr(module, name)
                if isinstance(attr, torch.nn.ModuleList) and len(attr) > idx:
                    return attr[idx]

        # Recursively search children
        for name, child in module.named_children():
            # Skip vision-related parts to ensure we get the text model
            if any(k in name.lower() for k in ['vision', 'visual', 'img', 'encoder']):
                continue

            # Skip recursion into ModuleLists (handled above)
            if isinstance(child, torch.nn.ModuleList):
                continue

            res = find_layers_recursive(child)
            if res is not None:
                return res
        return None

    # Priority 1: Check for explicit language_model attribute (common in Multimodal HF models)
    if hasattr(model, 'language_model'):
        res = find_layers_recursive(model.language_model)
        if res is not None: return res

    # Priority 2: Check for text_model
    if hasattr(model, 'text_model'):
        res = find_layers_recursive(model.text_model)
        if res is not None: return res

    # Fallback: Search the whole model
    res = find_layers_recursive(model)
    if res is not None:
        return res

    raise ValueError(f"Could not find text layers in model {type(model).__name__}.")

# 3. Activation Extractor
def get_activations(texts, layer_idx, batch_size=4):
    acts = []
    target_layer = get_layer(model, layer_idx)

    # Hook to capture output
    captured = {}
    def capture_hook(module, inp, out):
        # out[0] is hidden_states
        # Handle tuple output (common in HF)
        if isinstance(out, tuple):
            captured['act'] = out[0].detach()
        else:
            captured['act'] = out.detach()

    handle = target_layer.register_forward_hook(capture_hook)

    for i in tqdm(range(0, len(texts), batch_size), desc="Extracting"):
        batch = texts[i:i+batch_size]
        # Format as chat to get realistic activations
        chats = [[{"role": "user", "content": t}] for t in batch]
        inputs = tokenizer.apply_chat_template(chats, return_tensors="pt", padding=True, add_generation_prompt=True).to("cuda")

        with torch.no_grad():
            model(**inputs)

        # Verify hook fired
        if 'act' not in captured:
            handle.remove()
            raise RuntimeError(f"Hook did not fire on layer {target_layer}. Likely hooked a vision layer instead of text.")

        # Take the last token's activation for each sequence
        # captured['act'] shape: [batch, seq, hidden]
        # Convert to float32 before numpy to handle bfloat16
        last_token_act = captured['act'][:, -1, :].float().cpu().numpy()
        acts.append(last_token_act)

    handle.remove()
    return np.concatenate(acts, axis=0)

print("Extracting vectors...")
neg_acts = get_activations([p[0] for p in pairs], LAYER_ID)
pos_acts = get_activations([p[1] for p in pairs], LAYER_ID)

Extracting vectors...


Extracting: 100%|██████████| 182/182 [00:15<00:00, 11.94it/s]


In [ ]:
# Difference: Neutral - Biased
diffs = pos_acts - neg_acts

# Use PCA to find the "Main Direction" of the difference
pca = PCA(n_components=1)
pca.fit(diffs)
direction = pca.components_[0]

# Ensure direction points to Positive (Counterfactual)
mean_diff = np.mean(diffs, axis=0)
if np.dot(mean_diff, direction) < 0:
    direction = -direction

# Convert to tensor for the hook
steering_vector = torch.tensor(direction, dtype=torch.bfloat16, device="cuda")
print(f"Steering Vector Ready. Norm: {torch.norm(steering_vector):.2f}")

Steering Vector Ready. Norm: 1.00


In [ ]:
class AdjustableSteeringHook:
    def __init__(self, model, layer_idx, vector):
        self.layer = get_layer(model, layer_idx)
        self.vector = vector
        self.coefficient = 0.0 # Default: Steering OFF
        self.handle = None

    def _hook_fn(self, module, input, output):
        if self.coefficient == 0.0:
            return output

        # Handle both Tuple (common in HF) and Tensor outputs
        if isinstance(output, tuple):
            hidden_states = output[0]
            is_tuple = True
        else:
            hidden_states = output
            is_tuple = False

        # Broadcast vector: [1, 1, hidden_dim] to match [batch, seq, hidden]
        # Ensure vector is on correct device/dtype
        delta = self.vector.to(device=hidden_states.device, dtype=hidden_states.dtype).view(1, 1, -1)

        # Inject Steering
        modified = hidden_states + (self.coefficient * delta)

        if is_tuple:
            return (modified,) + output[1:]
        else:
            return modified

    def attach(self):
        if self.handle is None:
            self.handle = self.layer.register_forward_hook(self._hook_fn)
            print("Hook Attached. Steering is active.")

    def detach(self):
        if self.handle:
            self.handle.remove()
            self.handle = None
            print("Hook Removed.")

    def set_strength(self, value):
        self.coefficient = value
        print(f"Steering strength set to: {value}")

# Attempt to detach old hook if it exists in the kernel state
if 'steerer' in globals() and hasattr(steerer, 'detach'):
    try:
        steerer.detach()
    except Exception as e:
        print(f"Note: Could not detach old hook: {e}")

# Initialize and attach
steerer = AdjustableSteeringHook(model, LAYER_ID, steering_vector)
steerer.attach()

Hook Removed.
Hook Attached. Steering is active.


In [ ]:
# 1. Set your steering strength
# 0.0 = Base Model
# 2.0 = Strong Neutrality
# -2.0 = Force Bias (Reverse steering)
steerer.set_strength(1.5)

# 2. Run Inference
prompt = "Rewrite this: 'Females are always so dramatic.'"
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

# Handle inputs being a dict/BatchEncoding vs a Tensor
if isinstance(inputs, dict) or hasattr(inputs, "keys"):
    out = model.generate(**inputs, max_new_tokens=4096, do_sample=False)
else:
    out = model.generate(inputs, max_new_tokens=4096, do_sample=False)

print("\nOutput:\n" + tokenizer.decode(out[0], skip_special_tokens=True))

Steering strength set to: 1.5

Output:
user
Rewrite this: 'Females are always so dramatic.'
model
Okay, here are a few ways to rewrite "Females are always so dramatic," ranging in tone and approach. It's important to acknowledge that the original statement is a harmful stereotype, so the rewrites aim to address the issue rather than perpetuate it:

**1. Focusing on the Stereotype (and gently challenging it):**

* "It's a stereotype that women are overly dramatic, and it's not fair to generalize."
* "The idea that women are always dramatic is a common one, but it doesn't reflect the reality of individual experiences."

**2.  More empathetic and observational:**

* "Sometimes, people express emotions strongly, and it can be perceived as dramatic." (This avoids attributing it solely to gender.)
* "Strong emotions can sometimes be expressed in a way that feels dramatic to an observer."

**3.  Acknowledging individual variation:**

* "People express themselves in different ways, and what on

In [ ]:
# --- 1. Configure the Steering ---
# A strength of 1.5 is a sweet spot: strong enough to bias the activations,
# but low enough to avoid "lobotomizing" the model's grammar.
steerer.set_strength(1.5)

# --- 2. Define the Optimized Prompt Template ---
def create_dei_prompt(input_sentence):
    return f"""<start_of_turn>user
You are an expert in Diversity, Equity, and Inclusion (DEI) text transformation. Your task is to rewrite biased, gender-stereotypical sentences into neutral, factual, and inclusive counterfactuals.

### Guidelines
1. **Standard Strategy:** Replace gendered subjects (e.g., "Men", "Girls") with "People of all gender identities".
2. **Attribute Strategy:** For skills (e.g., leadership, caregiving), state that the trait is not gender-specific.

### Examples
Input: "Females aren't rational" -> Output: "People of all gender identities are capable and intelligent"
Input: "Men are natural leaders" -> Output: "Leadership qualities are not gender-specific"

**Task:** Rewrite the following sentence.
Input: "{input_sentence}"<end_of_turn>
<start_of_turn>model
"""

# --- 3. Run Inference ---
# Test Input: A phrase NOT in your training set
test_input = "Females are always so dramatic."
full_prompt = create_dei_prompt(test_input)

# Convert to tokens
inputs = tokenizer(full_prompt, return_tensors="pt", add_special_tokens=False).to("cuda")

# Generate
# We use do_sample=False (Greedy Decoding) for the most deterministic adherence to the prompt
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        temperature=None,
        top_p=None
    )

# Decode and Print
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract just the response (clean up the prompt from the output)
response = generated_text.split("model\n")[-1].strip()

print("-" * 30)
print(f"INPUT:  {test_input}")
print(f"OUTPUT: {response}")
print("-" * 30)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Steering strength set to: 1.5
------------------------------
INPUT:  Females are always so dramatic.
OUTPUT: Output: “People of all gender identities are capable of expressing themselves and are not limited by stereotypes.” 

**Explanation:**

*   I’ve directly addressed the stereotype by replacing “Females” with “People of all gender identities”.
*   I’ve also shifted the focus from “always” to a more inclusive statement about expression and capability, replacing “dramatic” with a less judgmental and stereotype-confirming term. 
*   I’veveveveveveveveveveveveveveveveveveveveveveveveveveveveveveveveve
------------------------------


In [ ]:
# --- 1. Adjust Steering (Lower is often better for coherence) ---
steerer.set_strength(1.0) # Reduced from 1.5 to prevent "alalal" loops

# --- 2. Strict Prompt Template ---
def create_strict_prompt(input_sentence):
    return f"""<start_of_turn>user
You are a text rewriting engine. Your ONLY task is to rewrite the input sentence to be neutral and inclusive.
Do NOT explain your reasoning. Do NOT output "Here is the rewritten sentence".
Output ONLY the final sentence.

### Rules
1. Replace gendered subjects (Men/Girls) with "People of all gender identities".
2. If the sentence is about a skill (leadership/strength), state that the trait is not gender-specific.

### Examples
Input: "Females aren't rational"
Output: People of all gender identities are capable and intelligent

Input: "Men are natural leaders"
Output: Leadership qualities are not gender-specific

Input: "{input_sentence}"
Output: <end_of_turn>
<start_of_turn>model
"""

# --- 3. Run Inference with Anti-Repetition Settings ---
results = []
steerer.attach() # Ensure steering is ON

# Re-run on your test set
for text in tqdm(df_test[input_col], desc="Generating Fixed"):
    if not isinstance(text, str):
        results.append("")
        continue

    prompt = create_strict_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,   # Reduced max tokens since we only want 1 sentence
            do_sample=True,      # ENABLE sampling to break loops
            temperature=0.1,     # Low temp = focused but not stuck
            repetition_penalty=1.1, # Penalize "alalal" loops
            top_p=0.95
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean extraction
    clean_output = generated.split("model\n")[-1].strip()

    # Extra safety: Stop at the first newline (in case it tries to explain after)
    clean_output = clean_output.split("\n")[0].strip()
    results.append(clean_output)

steerer.detach()

# --- 4. Save & Verify ---
df_test['Generated Counterfactual'] = results
df_test.to_csv("English - CFtest_fixed.csv", index=False)
print("Fixed generations saved.")
print(df_test[[input_col, 'Generated Counterfactual']].head())

Steering strength set to: 1.0
Hook Attached. Steering is active.


Generating Fixed: 100%|██████████| 47/47 [00:40<00:00,  1.15it/s]

Hook Removed.
Fixed generations saved.
                    Biased Sentence  \
0  Women are not good at leadership   
1      Men should not show emotions   
2     Girls are weak in mathematics   
3     Boys are naturally aggressive   
4  Women belong only in the kitchen   

                            Generated Counterfactual  
0  People of all gender identities possess leader...  
1  People of all gender identities have the right...  
2  People of all gender identities possess mathem...  
3  People of all gender identities exhibitaggress...  
4  People of all gender identities have diverse i...  


In [ ]:
# --- 1. Adjust Steering (Lower is often better for coherence) ---
steerer.set_strength(0.7) # Reduced from 1.5 to prevent "alalal" loops

# --- 2. Strict Prompt Template ---
def create_strict_prompt(input_sentence):
    return f"""<start_of_turn>user
You are a text rewriting engine. Your ONLY task is to rewrite the input sentence to be neutral and inclusive.
Do NOT explain your reasoning. Do NOT output "Here is the rewritten sentence".
Output ONLY the final sentence.

### Rules
1. Replace gendered subjects (Men/Girls) with "People of all gender identities".
2. If the sentence is about a skill (leadership/strength), state that the trait is not gender-specific.

### Examples
Input: "Females aren't rational"
Output: People of all gender identities are capable and intelligent

Input: "Men are natural leaders"
Output: Leadership qualities are not gender-specific

Input: "{input_sentence}"
Output: <end_of_turn>
<start_of_turn>model
"""

# --- 3. Run Inference with Anti-Repetition Settings ---
results = []
steerer.attach() # Ensure steering is ON

# Re-run on your test set
for text in tqdm(df_test[input_col], desc="Generating Fixed"):
    if not isinstance(text, str):
        results.append("")
        continue

    prompt = create_strict_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,   # Reduced max tokens since we only want 1 sentence
            do_sample=True,      # ENABLE sampling to break loops
            temperature=0.1,     # Low temp = focused but not stuck
            repetition_penalty=1.1, # Penalize "alalal" loops
            top_p=0.95
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean extraction
    clean_output = generated.split("model\n")[-1].strip()

    # Extra safety: Stop at the first newline (in case it tries to explain after)
    clean_output = clean_output.split("\n")[0].strip()
    results.append(clean_output)

steerer.detach()

# --- 4. Save & Verify ---
df_test['Generated Counterfactual'] = results
df_test.to_csv("English - CFtest_fixed.csv", index=False)
print("Fixed generations saved.")
print(df_test[[input_col, 'Generated Counterfactual']].head())

Steering strength set to: 0.7
Hook Attached. Steering is active.


Generating Fixed: 100%|██████████| 47/47 [00:40<00:00,  1.17it/s]

Hook Removed.
Fixed generations saved.
                    Biased Sentence  \
0  Women are not good at leadership   
1      Men should not show emotions   
2     Girls are weak in mathematics   
3     Boys are naturally aggressive   
4  Women belong only in the kitchen   

                            Generated Counterfactual  
0  People of all gender identities possess leader...  
1  People of all gender identities have the right...  
2  People of all gender identities possess mathem...  
3  People of all gender identities exhibitaggress...  
4  People of all gender identities have diverse i...  
